# 0. Importing PyTorch and setting up device

In [1]:
import os
from PIL import Image

import torch
from torch import nn
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchmetrics import F1Score, Recall, Precision
from torchmetrics.classification import (
    MulticlassPrecision,
    MulticlassRecall,
    MulticlassF1Score
)
print(torch.__version__)

2.3.0+cu118


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
#device = 'cpu'
device

'cuda'

# 1. Data preparation and reading

In [133]:
#Load Classes
with open(r"E:\Courses\AI\Deep Learning\food-101\meta/classes.txt", "r") as f:
    classes = [line.strip() for line in f if line.strip()]
selected_classes = classes[:30]
class_to_idx = {c: i for i, c in enumerate(selected_classes)}

print("Num classes:", len(class_to_idx))

Num classes: 30


In [5]:
#Dataset Clas
class Food101Dataset(Dataset):

    def __init__(self, images_dir, split_file, class_to_idx, transform=None):
        self.images_dir = images_dir
        self.transform = transform
        self.class_to_idx = class_to_idx

        with open(split_file, "r") as f:
            all_samples = [line.strip() for line in f if line.strip()]

        self.samples = [
            s for s in all_samples
            if s.split("/")[0] in class_to_idx
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        sample = self.samples[idx]  # e.g. apple_pie/1005649
        class_name = sample.split("/")[0]

        image_path = os.path.join(self.images_dir, sample + ".jpg")

        image = Image.open(image_path).convert("RGB")
        label = self.class_to_idx[class_name]

        if self.transform:
            image = self.transform(image)

        return image, label

# 2. Transforming data 

In [ ]:
# ResNet50 & EfficientNet-B0
train_transforms = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# AdvancedFoodCNN
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Create Datasets

In [9]:
train_dataset = Food101Dataset(
    images_dir="food-101/images",
    split_file="food-101/meta/train.txt",
    class_to_idx=class_to_idx,
    transform=train_transforms
)

test_dataset = Food101Dataset(
    images_dir="food-101/images",
    split_file="food-101/meta/test.txt",
    class_to_idx=class_to_idx,
    transform=test_transforms
)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 22500
Test size: 7500


# 4. Turn loaded images into DataLoader's

In [17]:
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [19]:
# Test One Batch
images, labels = next(iter(train_loader))

print("Batch images:", images.shape)
print("Batch labels:", labels.shape)

Batch images: torch.Size([16, 3, 224, 224])
Batch labels: torch.Size([16])


In [20]:
# Test single sample
img, label = train_dataset[0]

print("Single image shape:", img.shape)
print("Label:", label)

Single image shape: torch.Size([3, 224, 224])
Label: 0


# 7 Create train and test loops functions

In [22]:
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

In [23]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               scheduler,
               accuracy_fn,
               device: torch.device,
               num_classes: int = 30,
               task: str = "multiclass"):

    """Performs a training step."""

    total_batches = len(data_loader)

    # طباعة التقدم كل 10%
    progress_step = max(1, total_batches // 10)
    last_percent = -1

    # Metrics
    precision_metric = MulticlassPrecision(
        num_classes=num_classes,
        average="macro"
    ).to(device)

    recall_metric = MulticlassRecall(
        num_classes=num_classes,
        average="macro"
    ).to(device)

    f1_metric = MulticlassF1Score(
        num_classes=num_classes,
        average="macro"
    ).to(device)

    train_loss = 0
    train_acc = 0

    model.train()

    for batch, (X, y) in enumerate(data_loader):

        percent = int((batch / total_batches) * 100)

        if percent % 10 == 0 and percent != last_percent:
            print(f"📊 Progress: {percent}% ({batch}/{total_batches})")
            last_percent = percent

        X, y = X.to(device), y.to(device)

        # Forward
        y_pred_logits = model(X)

        # Predictions
        y_pred_labels = torch.argmax(y_pred_logits, dim=1)

        # Loss
        loss = loss_fn(y_pred_logits, y)

        train_loss += loss.item()
        train_acc += accuracy_fn(y_true=y, y_pred=y_pred_labels)

        # Update metrics
        precision_metric.update(y_pred_labels, y)
        recall_metric.update(y_pred_labels, y)
        f1_metric.update(y_pred_labels, y)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Averages
    train_loss /= total_batches
    train_acc /= total_batches

    # Compute metrics on full epoch
    total_precision = precision_metric.compute().item()
    total_recall = recall_metric.compute().item()
    total_f1 = f1_metric.compute().item()

    # Reset
    precision_metric.reset()
    recall_metric.reset()
    f1_metric.reset()

    print(f"📊 Progress: 100% ({total_batches}/{total_batches})")
    print(f"Train loss: {train_loss:.5f} | Train acc: {train_acc:.2f}%")
    print(
        f"Train Precision: {total_precision:.4f} | "
        f"Train Recall: {total_recall:.4f} | "
        f"Train F1-Score: {total_f1:.4f}"
    )

    return (
        train_loss,
        train_acc,
        total_precision,
        total_recall,
        total_f1
    )

In [24]:
def test_step(model: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device: torch.device,
              num_classes: int = 30,
              task: str = "multiclass"):

    """Performs a testing loop step on model going over data_loader."""

    total_batches = len(data_loader)

    precision_metric = MulticlassPrecision(
        num_classes=num_classes,
        average="macro"
    ).to(device)

    recall_metric = MulticlassRecall(
        num_classes=num_classes,
        average="macro"
    ).to(device)

    f1_metric = MulticlassF1Score(
        num_classes=num_classes,
        average="macro"
    ).to(device)

    test_loss = 0
    test_acc = 0

    model.eval()

    with torch.inference_mode():

        for X, y in data_loader:

            X, y = X.to(device), y.to(device)

            # Forward
            test_pred_logits = model(X)

            # Predictions
            test_pred_labels = torch.argmax(test_pred_logits, dim=1)

            # Loss
            loss = loss_fn(test_pred_logits, y)

            test_loss += loss.item()

            test_acc += accuracy_fn(
                y_true=y,
                y_pred=test_pred_labels
            )

            # Update metrics
            precision_metric.update(test_pred_labels, y)
            recall_metric.update(test_pred_labels, y)
            f1_metric.update(test_pred_labels, y)

    test_loss /= total_batches
    test_acc /= total_batches

    total_precision = precision_metric.compute().item()
    total_recall = recall_metric.compute().item()
    total_f1 = f1_metric.compute().item()

    precision_metric.reset()
    recall_metric.reset()
    f1_metric.reset()

    print(
        f"Test loss: {test_loss:.5f} | "
        f"Test acc: {test_acc:.2f}%"
    )

    print(
        f"Test Precision: {total_precision:.4f} | "
        f"Test Recall: {total_recall:.4f} | "
        f"Test F1-Score: {total_f1:.4f}\n"
    )

    return (
        test_loss,
        test_acc,
        total_precision,
        total_recall,
        total_f1
    )

In [25]:
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
import torch
import os
from datetime import datetime

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          scheduler,
          accuracy_fn,  
          loss_fn: torch.nn.Module = torch.nn.CrossEntropyLoss(),
          epochs: int = 5, 
          device: torch.device = "cuda" if torch.cuda.is_available() else "cpu",
          num_classes: int = 10,       
          task: str = "multiclass",
          patience: int = 3,            
          min_delta: float = 0.0,       
          save_path: str = "best_model.pth",
          experiment_name: str = "food30_run"): 
  
  # --- 1. تم الإصلاح هنا: استخدام %S الكبيرة وتغيير النقطتين لفواصل آمنة للمجلدات ---
  current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
  log_dir = os.path.join("runs", experiment_name, current_time)
  writer = SummaryWriter(log_dir=log_dir)
  print(f"🚀 Created a new local run at: '{log_dir}'")

  results = {
      "train_loss": [], "train_acc": [], "train_precision": [], "train_recall": [], "train_f1": [],
      "test_loss": [], "test_acc": [], "test_precision": [], "test_recall": [], "test_f1": []
  }
  
  best_loss = float('inf')  
  patience_counter = 0      

  # 3. حلقة الـ Epochs
  for epoch in tqdm(range(epochs)):
    print(f"\n--- Epoch {epoch+1}/{epochs} ---")
    
    train_loss, train_acc, train_precision, train_recall, train_f1 = train_step(
        model=model, data_loader=train_dataloader, loss_fn=loss_fn,
        optimizer=optimizer, scheduler=scheduler, accuracy_fn=accuracy_fn,       
        device=device, num_classes=num_classes, task=task
    )
    
    test_loss, test_acc, test_precision, test_recall, test_f1 = test_step(
        model=model, data_loader=test_dataloader, loss_fn=loss_fn,
        accuracy_fn=accuracy_fn, device=device, num_classes=num_classes, task=task
    )
    
    if scheduler is not None:
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current Learning Rate: {current_lr}")
        
    print(f"Summary -> Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

    # حفظ المقاييس في الـ Run
    writer.add_scalar('Loss/Train', train_loss, epoch)
    writer.add_scalar('Loss/Test', test_loss, epoch)
    writer.add_scalar('Accuracy/Train', train_acc, epoch)
    writer.add_scalar('Accuracy/Test', test_acc, epoch)
    writer.add_scalar('F1-Score/Train', train_f1, epoch)
    writer.add_scalar('F1-Score/Test', test_f1, epoch)

    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["train_precision"].append(train_precision)
    results["train_recall"].append(train_recall)
    results["train_f1"].append(train_f1)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)
    results["test_precision"].append(test_precision)
    results["test_recall"].append(test_recall)
    results["test_f1"].append(test_f1)
    
    # منطق الـ Early Stopping
    if test_loss < (best_loss - min_delta):
        best_loss = test_loss
        patience_counter = 0  
        torch.save(obj=model.state_dict(), f=save_path)
        print(f"🎯 Test Loss improved! Saving best model weights to: '{save_path}'")
    else:
        patience_counter += 1  
        print(f"⚠️ No improvement in Test Loss. EarlyStopping Counter: {patience_counter}/{patience}")
        
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping triggered at Epoch {epoch+1}! Training stopped.")
            break  
            
  writer.close()
  return results

# GPU cleaning and memory free-up

In [ ]:
import gc
import torch

if 'model_resnet' in locals():
    del model_resnet

gc.collect()

torch.cuda.empty_cache()

print("GPU Memory has been successfully cleared and ready for training!")

# Train EfficientNet-B0 Model with size 224

In [32]:
import torch
import torchvision
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# 1. جلب الأوزان 
weights = EfficientNet_B0_Weights.DEFAULT
model_2 = efficientnet_b0(weights=weights)

# 2. تجميد الطبقاتا
for param in model_2.parameters():
    param.requires_grad = False

# 3. تعديل الطبقة الأخيرة 
model_2.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280, out_features=30) # 30 هو عدد الأصناف لديك
)

# 4. إرسال الموديل إلى GPU
model_2 = model_2.to(device)

In [37]:
# Set random seeds
torch.manual_seed(42) 
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 25

# Setup loss function and optimizer 
loss_fn = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(params=model_2.parameters(),
                             lr=0.001)
scheduler =  torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
# Start the timer
from timeit import default_timer as timer
start_time = timer() 
# Train model_0 (قُمنا بإضافة البارامترات المفقودة هنا)
model_3_results = train(model=model_2,
                        train_dataloader=train_loader,
                        test_dataloader=test_loader,
                        optimizer=optimizer,
                        scheduler=None,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device,
                        accuracy_fn=accuracy_fn, 
                        num_classes
                        = len(class_to_idx), 
                        task="multiclass",
                        patience=3,
                        min_delta=0.005,
                        save_path="models/best_food30_model.pth",
                        experiment_name="EfficientNet_FineTuning")

# End the timer and print out how long it took
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

🚀 Created a new local run at: 'runs\EfficientNet_FineTuning\2026-06-16_23-43-38'


  0%|          | 0/25 [00:00<?, ?it/s]


--- Epoch 1/25 ---
📊 Progress: 0% (0/704)
📊 Progress: 10% (71/704)
📊 Progress: 20% (141/704)
📊 Progress: 30% (212/704)
📊 Progress: 40% (282/704)
📊 Progress: 50% (352/704)
📊 Progress: 60% (423/704)
📊 Progress: 70% (493/704)
📊 Progress: 80% (564/704)
📊 Progress: 90% (634/704)
📊 Progress: 100% (704/704)
Train loss: 2.13542 | Train acc: 43.73%
Train Precision: 0.4289 | Train Recall: 0.4372 | Train F1-Score: 0.4289
Test loss: 1.34531 | Test acc: 64.03%
Test Precision: 0.6412 | Test Recall: 0.6393 | Test F1-Score: 0.6318

Summary -> Train Loss: 2.1354 | Train Acc: 43.73% | Test Loss: 1.3453 | Test Acc: 64.03%
🎯 Test Loss improved! Saving best model weights to: 'models/best_food30_model.pth'

--- Epoch 2/25 ---
📊 Progress: 0% (0/704)
📊 Progress: 10% (71/704)
📊 Progress: 20% (141/704)
📊 Progress: 30% (212/704)
📊 Progress: 40% (282/704)
📊 Progress: 50% (352/704)
📊 Progress: 60% (423/704)
📊 Progress: 70% (493/704)
📊 Progress: 80% (564/704)
📊 Progress: 90% (634/704)
📊 Progress: 100% (704/704)
Tr

In [43]:
print("="*50)
print("TRAIN SET EVALUATION")
print("="*50)

train_results = test_step(
    model=model_2,
    data_loader=train_loader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device,
    num_classes=len(class_to_idx)
)
print("="*50)
print("TEST SET EVALUATION")
print("="*50)

test_results = test_step(
    model=model_2,
    data_loader=test_loader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device,
    num_classes=len(class_to_idx)
)

TRAIN SET EVALUATION
Test loss: 1.16944 | Test acc: 66.86%
Test Precision: 0.6696 | Test Recall: 0.6682 | Test F1-Score: 0.6667

TEST SET EVALUATION
Test loss: 1.01473 | Test acc: 70.69%
Test Precision: 0.7074 | Test Recall: 0.7061 | Test F1-Score: 0.7036



# Train ResNet50 Model

In [30]:
from torchvision.models import resnet50, ResNet50_Weights

resnet_weights = ResNet50_Weights.DEFAULT
model_resnet = resnet50(weights=None)
state_dict = torch.load("resnet50.pth", map_location=device)
model_resnet.load_state_dict(state_dict)

for param in model_resnet.parameters():
    param.requires_grad = False

in_features = model_resnet.fc.in_features

model_resnet.fc = torch.nn.Sequential(
    torch.nn.Dropout(p=0.3),
    torch.nn.Linear(in_features=in_features, out_features=30)
)

model_resnet = model_resnet.to(device)

print("✅ ResNet50 Backbone frozen and FC layer adjusted to 30 outputs successfully!")

✅ ResNet50 Backbone frozen and FC layer adjusted to 30 outputs successfully!


In [40]:
torch.manual_seed(42) 
torch.cuda.manual_seed(42)

NUM_EPOCHS = 25

loss_fn = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(params=model_resnet.parameters(), lr=0.001)
scheduler =  torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

from timeit import default_timer as timer
start_time = timer() 

model_3_results = train(model=model_resnet,
                        train_dataloader=train_loader,
                        test_dataloader=test_loader,
                        optimizer=optimizer,
                        scheduler=None,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device,
                        accuracy_fn=accuracy_fn, 
                        num_classes
                        = len(class_to_idx), 
                        task="multiclass",
                        patience=5,
                        min_delta=0.005,
                        save_path="models/model_resnet.pth",
                        experiment_name="ResNet50_FineTuning")

end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

🚀 Created a new local run at: 'runs\ResNet50_FineTuning\2026-06-17_01-59-59'


  0%|          | 0/25 [00:00<?, ?it/s]


--- Epoch 1/25 ---
📊 Progress: 0% (0/704)
📊 Progress: 10% (71/704)
📊 Progress: 20% (141/704)
📊 Progress: 30% (212/704)
📊 Progress: 40% (282/704)
📊 Progress: 50% (352/704)
📊 Progress: 60% (423/704)
📊 Progress: 70% (493/704)
📊 Progress: 80% (564/704)
📊 Progress: 90% (634/704)


C:\Users\aa\anaconda3\envs\gpu_env\lib\site-packages\torch\nn\modules\conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ..\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,


📊 Progress: 100% (704/704)
Train loss: 2.29959 | Train acc: 42.67%
Train Precision: 0.4202 | Train Recall: 0.4269 | Train F1-Score: 0.4186
Test loss: 1.50719 | Test acc: 65.81%
Test Precision: 0.6700 | Test Recall: 0.6579 | Test F1-Score: 0.6499

Summary -> Train Loss: 2.2996 | Train Acc: 42.67% | Test Loss: 1.5072 | Test Acc: 65.81%
🎯 Test Loss improved! Saving best model weights to: 'models/model_resnet.pth'

--- Epoch 2/25 ---
📊 Progress: 0% (0/704)
📊 Progress: 10% (71/704)
📊 Progress: 20% (141/704)
📊 Progress: 30% (212/704)
📊 Progress: 40% (282/704)
📊 Progress: 50% (352/704)
📊 Progress: 60% (423/704)
📊 Progress: 70% (493/704)
📊 Progress: 80% (564/704)
📊 Progress: 90% (634/704)
📊 Progress: 100% (704/704)
Train loss: 1.64348 | Train acc: 57.29%
Train Precision: 0.5683 | Train Recall: 0.5733 | Train F1-Score: 0.5686
Test loss: 1.20261 | Test acc: 70.31%
Test Precision: 0.7205 | Test Recall: 0.7029 | Test F1-Score: 0.6989

Summary -> Train Loss: 1.6435 | Train Acc: 57.29% | Test Loss: 

In [41]:
print("="*50)
print("TRAIN SET EVALUATION")
print("="*50)

train_results = test_step(
    model=model_resnet,
    data_loader=train_loader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device,
    num_classes=len(class_to_idx)
)
print("="*50)
print("TEST SET EVALUATION")
print("="*50)

test_results = test_step(
    model=model_resnet,
    data_loader=test_loader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device,
    num_classes=len(class_to_idx)
)

TRAIN SET EVALUATION
Test loss: 0.76583 | Test acc: 78.36%
Test Precision: 0.7861 | Test Recall: 0.7836 | Test F1-Score: 0.7827

TEST SET EVALUATION
Test loss: 0.71820 | Test acc: 79.18%
Test Precision: 0.7911 | Test Recall: 0.7915 | Test F1-Score: 0.7892



# Train EfficientNet-B0 Model with size 512

In [43]:
import torch
import torchvision
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# 1. جلب الأوزان 
weights = EfficientNet_B0_Weights.DEFAULT
model_efficien = efficientnet_b0(weights=weights)

# 2. تجميد الطبقاتا
for param in model_efficien.parameters():
    param.requires_grad = False

# 3. تعديل الطبقة الأخيرة 
model_efficien.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280, out_features=30) # 30 هو عدد الأصناف لديك
)

# 4. إرسال الموديل إلى GPU
model_efficien = model_efficien.to(device)

In [44]:
# Set random seeds
torch.manual_seed(42) 
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 25

# Setup loss function and optimizer 
loss_fn = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(params=model_efficien.parameters(),
                             lr=0.001)
scheduler =  torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
# Start the timer
from timeit import default_timer as timer
start_time = timer() 
# Train model_0 (قُمنا بإضافة البارامترات المفقودة هنا)
model_3_results = train(model=model_efficien,
                        train_dataloader=train_loader,
                        test_dataloader=test_loader,
                        optimizer=optimizer,
                        scheduler=None,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device,
                        accuracy_fn=accuracy_fn, 
                        num_classes
                        = len(class_to_idx), 
                        task="multiclass",
                        patience=3,
                        min_delta=0.005,
                        save_path="models/model_efficien.pth",
                        experiment_name="EfficientNet_FineTuning")

# End the timer and print out how long it took
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

🚀 Created a new local run at: 'runs\EfficientNet_FineTuning\2026-06-17_14-59-45'


  0%|          | 0/25 [00:00<?, ?it/s]


--- Epoch 1/25 ---
📊 Progress: 0% (0/704)
📊 Progress: 10% (71/704)
📊 Progress: 20% (141/704)
📊 Progress: 30% (212/704)
📊 Progress: 40% (282/704)
📊 Progress: 50% (352/704)
📊 Progress: 60% (423/704)
📊 Progress: 70% (493/704)
📊 Progress: 80% (564/704)
📊 Progress: 90% (634/704)
📊 Progress: 100% (704/704)
Train loss: 2.10379 | Train acc: 46.94%
Train Precision: 0.4625 | Train Recall: 0.4693 | Train F1-Score: 0.4619
Test loss: 1.22343 | Test acc: 68.79%
Test Precision: 0.6900 | Test Recall: 0.6873 | Test F1-Score: 0.6791

Summary -> Train Loss: 2.1038 | Train Acc: 46.94% | Test Loss: 1.2234 | Test Acc: 68.79%
🎯 Test Loss improved! Saving best model weights to: 'models/best_food30_model.pth'

--- Epoch 2/25 ---
📊 Progress: 0% (0/704)
📊 Progress: 10% (71/704)
📊 Progress: 20% (141/704)
📊 Progress: 30% (212/704)
📊 Progress: 40% (282/704)
📊 Progress: 50% (352/704)
📊 Progress: 60% (423/704)
📊 Progress: 70% (493/704)
📊 Progress: 80% (564/704)
📊 Progress: 90% (634/704)
📊 Progress: 100% (704/704)
Tr

In [45]:
print("="*50)
print("TRAIN SET EVALUATION")
print("="*50)

train_results = test_step(
    model=model_efficien,
    data_loader=train_loader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device,
    num_classes=len(class_to_idx)
)
print("="*50)
print("TEST SET EVALUATION")
print("="*50)

test_results = test_step(
    model=model_efficien,
    data_loader=test_loader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device,
    num_classes=len(class_to_idx)
)

TRAIN SET EVALUATION
Test loss: 0.92422 | Test acc: 73.14%
Test Precision: 0.7350 | Test Recall: 0.7314 | Test F1-Score: 0.7302

TEST SET EVALUATION
Test loss: 0.76539 | Test acc: 77.41%
Test Precision: 0.7764 | Test Recall: 0.7735 | Test F1-Score: 0.7717



# 5. Create model_0 class

In [32]:
class AdvancedFoodCNN(nn.Module):
    def __init__(self, input_shape: int, output_shape: int):
        super().__init__()
        
        # Block 1
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32), 
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) 
        )
        
        # Block 1
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Block 1
        self.conv_block_3 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=256 * 7 * 7, out_features=256),
            nn.ReLU(),
            nn.Dropout(p=0.1), 
            nn.Linear(in_features=256, out_features=output_shape)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.conv_block_3(x)
        x = self.adaptive_pool(x)
        x = self.classifier(x)
        return x

In [34]:
torch.manual_seed(42)
AdvancedFoodCNN = AdvancedFoodCNN(input_shape=3, output_shape=30).to(device)

AdvancedFoodCNN

AdvancedFoodCNN(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_3): Seque

# 8 Train and evaluate model_0

In [48]:
# Set random seeds
torch.manual_seed(42) 
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 25

# Setup loss function and optimizer 
loss_fn = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(params=AdvancedFoodCNN.parameters(),
                             lr=0.0001)
scheduler =  torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
# Start the timer
from timeit import default_timer as timer
start_time = timer() 

# Train model_0 (قُمنا بإضافة البارامترات المفقودة هنا)
model_0_results = train(model=AdvancedFoodCNN,
                        train_dataloader=train_loader,
                        test_dataloader=test_loader,
                        optimizer=optimizer,
                        scheduler=None,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device,
                        accuracy_fn=accuracy_fn, 
                        num_classes= len(class_to_idx), 
                        task="multiclass",
                        patience=5,
                        min_delta=0.005,
                        save_path="models/AdvancedFoodCNN.pth",
                        experiment_name="AdvancedFoodCNN") 

# End the timer and print out how long it took
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

🚀 Created a new local run at: 'runs\AdvancedFoodCNN\2026-06-18_12-00-23'


  0%|          | 0/25 [00:00<?, ?it/s]


--- Epoch 1/25 ---
📊 Progress: 0% (0/1407)
📊 Progress: 10% (141/1407)
📊 Progress: 20% (282/1407)
📊 Progress: 30% (423/1407)
📊 Progress: 40% (563/1407)
📊 Progress: 50% (704/1407)
📊 Progress: 60% (845/1407)
📊 Progress: 70% (985/1407)
📊 Progress: 80% (1126/1407)
📊 Progress: 90% (1267/1407)
📊 Progress: 100% (1407/1407)
Train loss: 1.70593 | Train acc: 48.45%
Train Precision: 0.4813 | Train Recall: 0.4844 | Train F1-Score: 0.4741
Test loss: 1.73255 | Test acc: 49.88%
Test Precision: 0.4868 | Test Recall: 0.4989 | Test F1-Score: 0.4891

Summary -> Train Loss: 1.7059 | Train Acc: 48.45% | Test Loss: 1.7326 | Test Acc: 49.88%
🎯 Test Loss improved! Saving best model weights to: 'models/AdvancedFoodCNN.pth'

--- Epoch 2/25 ---
📊 Progress: 0% (0/1407)
📊 Progress: 10% (141/1407)
📊 Progress: 20% (282/1407)
📊 Progress: 30% (423/1407)
📊 Progress: 40% (563/1407)
📊 Progress: 50% (704/1407)
📊 Progress: 60% (845/1407)
📊 Progress: 70% (985/1407)
📊 Progress: 80% (1126/1407)
📊 Progress: 90% (1267/1407)
📊 P

In [51]:
print("="*50)
print("TRAIN SET EVALUATION")
print("="*50)

train_results = test_step(
    model=AdvancedFoodCNN,
    data_loader=train_loader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device,
    num_classes=len(class_to_idx)
)
print("="*50)
print("TEST SET EVALUATION")
print("="*50)

test_results = test_step(
    model=AdvancedFoodCNN,
    data_loader=test_loader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device,
    num_classes=len(class_to_idx)
)

TRAIN SET EVALUATION
Test loss: 1.28398 | Test acc: 63.21%
Test Precision: 0.6287 | Test Recall: 0.6319 | Test F1-Score: 0.6272

TEST SET EVALUATION
Test loss: 1.75247 | Test acc: 50.16%
Test Precision: 0.4952 | Test Recall: 0.5016 | Test F1-Score: 0.4951



# 6. Try a forward pass on a single image (to test the model)

In [19]:
image_batch, label_batch = next(iter(train_loader))
image_batch.shape, label_batch.shape

(torch.Size([10, 3, 224, 224]), torch.Size([10]))

In [20]:
# Try a forward pass
model_0(image_batch.to(device))

C:\Users\aa\anaconda3\envs\gpu_env\lib\site-packages\torch\nn\modules\conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ..\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,


tensor([[ 0.2592, -0.1619, -0.6911,  ...,  0.6379,  0.2063,  0.1579],
        [ 0.1510, -0.1710, -0.1784,  ...,  0.3937,  0.0989, -0.0535],
        [ 0.0364, -0.4683, -0.2037,  ...,  0.1135, -0.0449,  0.0538],
        ...,
        [ 0.1759, -0.2965, -0.3969,  ...,  0.2717,  0.3541,  0.1048],
        [-0.2860, -0.0971, -0.6141,  ...,  0.3843,  0.1655, -0.1018],
        [ 0.2754,  0.0029, -1.1142,  ...,  0.2984,  0.0768,  0.1056]],
       device='cuda:0', grad_fn=<AddmmBackward0>)

In [21]:
from torchinfo import summary
summary(model_0, input_size=[1, 3, 224, 224])

Layer (type:depth-idx)                   Output Shape              Param #
AdvancedFoodCNN                          [1, 101]                  --
├─Sequential: 1-1                        [1, 64, 112, 112]         --
│    └─Conv2d: 2-1                       [1, 32, 224, 224]         896
│    └─BatchNorm2d: 2-2                  [1, 32, 224, 224]         64
│    └─ReLU: 2-3                         [1, 32, 224, 224]         --
│    └─Conv2d: 2-4                       [1, 64, 224, 224]         18,496
│    └─BatchNorm2d: 2-5                  [1, 64, 224, 224]         128
│    └─ReLU: 2-6                         [1, 64, 224, 224]         --
│    └─MaxPool2d: 2-7                    [1, 64, 112, 112]         --
├─Sequential: 1-2                        [1, 128, 56, 56]          --
│    └─Conv2d: 2-8                       [1, 128, 112, 112]        73,856
│    └─BatchNorm2d: 2-9                  [1, 128, 112, 112]        256
│    └─ReLU: 2-10                        [1, 128, 112, 112]        --
│   

# Test the Models

In [102]:
import torch
from torch import nn
from torchvision import datasets, models, transforms

from PIL import Image
import os

In [104]:
def Prediction(model, transforme, image_path):
    class_names = [
    "apple_pie", "baby_back_ribs", "baklava", "beef_carpaccio", "beef_tartare",
    "beet_salad", "beignets", "bibimbap", "bread_pudding", "breakfast_burrito",
    "bruschetta", "caesar_salad", "cannoli", "caprese_salad", "carrot_cake",
    "ceviche", "cheesecake", "cheese_plate", "chicken_curry", "chicken_quesadilla",
    "chicken_wings", "chocolate_cake", "chocolate_mousse", "churros", "clam_chowder",
    "club_sandwich", "crab_cakes", "creme_brulee", "croque_madame", "cup_cakes"
    ]
    
    if os.path.exists(image_path):
        img = Image.open(image_path).convert('RGB')
    
        input_tensor = transform(img).unsqueeze(0).to(device)
    
        model.eval()
    
        with torch.no_grad():
            logist = model(input_tensor)
        
            probabilities = torch.nn.functional.softmax(logist[0], dim=0)
            predicted_idx = torch.argmax(probabilities).item()
        
        predicted_class = class_names[predicted_idx]
        confidence = probabilities[predicted_idx].item() * 100
    
        print(f"Prediction: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")
    else:
        print(f":The image file could not be found in the required path. {image_path}")

## Test Resnet

In [107]:
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [109]:
model_resnet = models.resnet50(weights=None) 

in_features = model_resnet.fc.in_features
model_resnet.fc = torch.nn.Sequential(
    torch.nn.Dropout(p=0.3),
    torch.nn.Linear(in_features=in_features, out_features=30)
)
device = "cuda" if torch.cuda.is_available else "cpu"
model_resnet.load_state_dict(torch.load(r".\Models\model_resnet.pth"))
print("Local weights loaded successfully.")
model_resnet = model_resnet.to(device)

Local weights loaded successfully.


In [110]:
Prediction(model_resnet, transform, "Test image/chocolate_cake.jpg")

Prediction: chocolate_cake
Confidence: 87.36%


In [111]:
Prediction(model_resnet, transform, "Test image/chicken_curry.webp")

Prediction: chicken_curry
Confidence: 47.95%


In [112]:
Prediction(model_resnet, transform, "Test image/bruschetta.webp")

Prediction: bruschetta
Confidence: 98.22%


## Test AdvancedFoodCNN

In [114]:
transform_AdvancedFoodCNN = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [115]:
model_AdvancedFoodCNN = AdvancedFoodCNN(input_shape=3, output_shape=30).to(device)
model_AdvancedFoodCNN.load_state_dict(torch.load(r".\Models\AdvancedFoodCNN.pth"))

<All keys matched successfully>

In [116]:
Prediction(model_AdvancedFoodCNN, transform_AdvancedFoodCNN, "Test image/chocolate_cake.jpg")

Prediction: chocolate_mousse
Confidence: 30.10%


In [117]:
Prediction(model_AdvancedFoodCNN, transform_AdvancedFoodCNN, "Test image/chicken_curry.webp")

Prediction: beef_tartare
Confidence: 27.32%


In [118]:
Prediction(model_AdvancedFoodCNN, transform_AdvancedFoodCNN, "Test image/bruschetta.webp")

Prediction: caprese_salad
Confidence: 25.13%
